# ASMR Helper · Colab 语音识别

在 Colab 选择 GPU 运行时（如 T4），按顺序执行单元格。首次运行会准备 Faster-Whisper 和对应模型，上传一段音频后输出 `result.srt`。把字幕带回 ASMR Helper，和原音频一起导入，或在 `asmr_prepare_draft` 里显式指定 `subtitle_path`。

本笔记本也可由 `scripts/run_colab_asr.py` 提交给已有的 Colab CLI 会话。识别流程输出带时间轴的原文字幕。可选环境单元格为 GPU Worker 的分离、语音合成和变声任务准备运行环境。

In [ ]:
# @title 1. 参数和 GPU
import os
from pathlib import Path
import subprocess
import sys
import tempfile

MODEL = "small"  # tiny、base、small、medium 或 large-v3
LANGUAGE = "ja"  # ja、zh、en、ko 或 auto
AUDIO_PATH = ""  # 留空后手动上传，或填运行时内现有音频的路径
PERSIST_MODEL_ON_DRIVE = False  # 设为 True 可把模型权重缓存到 Google Drive

cli_job_dir = os.environ.get("ASMR_COLAB_JOB_DIR")
if cli_job_dir:
    MODEL = os.environ.get("ASMR_COLAB_MODEL", MODEL)
    LANGUAGE = os.environ.get("ASMR_COLAB_LANGUAGE", LANGUAGE)
    job_dir = Path(cli_job_dir)
else:
    job_dir = Path(tempfile.mkdtemp(prefix="asmr-asr-", dir="/content"))

if MODEL not in {"tiny", "base", "small", "medium", "large-v3"}:
    raise ValueError("不支持的 Faster-Whisper 模型")
if LANGUAGE not in {"ja", "zh", "en", "ko", "auto"}:
    raise ValueError("不支持的语言设置")

try:
    subprocess.run(["nvidia-smi", "-L"], check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("请在 Colab 运行时设置中选择 GPU，再重新运行。") from exc

In [ ]:
# @title 2. 安装识别环境
import sysconfig

REPO_URL = "https://github.com/Flysky701/AsmrHelper.git"
REVISION = os.environ.get("ASMR_COLAB_REVISION", "6c3e30990ebfbe9161cd1839234986a73f73a812")
repo_dir = Path("/content") / f"AsmrHelper-{REVISION[:12]}"
if not (repo_dir / "packages/gpu_worker/cli.py").is_file():
    subprocess.run(["git", "init", "-q", str(repo_dir)], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", REPO_URL, REVISION], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    "faster-whisper==1.2.1", "ctranslate2==4.8.1", "nvidia-cublas-cu12",
    "nvidia-cudnn-cu12>=9,<10", "numpy>=1.24", "pydantic>=2,<3", "soundfile>=0.12",
    "pyyaml>=6", "imageio-ffmpeg>=0.4",
], check=True)

drive_models = Path("/content/drive/MyDrive/AsmrHelper_Models/models")
cache_root = Path("/content/AsmrHelperModels")
if PERSIST_MODEL_ON_DRIVE and not cli_job_dir:
    from google.colab import drive
    drive.mount("/content/drive")
    cache_root = drive_models
elif cli_job_dir and drive_models.is_dir():
    cache_root = drive_models
(cache_root / "whisper").mkdir(parents=True, exist_ok=True)

nvidia = Path(sysconfig.get_path("purelib")) / "nvidia"
library_dirs = [str(nvidia / name / "lib") for name in ("cublas", "cudnn")]
if os.environ.get("LD_LIBRARY_PATH"):
    library_dirs.append(os.environ["LD_LIBRARY_PATH"])
worker_env = {
    **os.environ,
    "LD_LIBRARY_PATH": os.pathsep.join(library_dirs),
    "ASMR_HELPER_COMPUTE": "cuda",
    "ASMR_HELPER_MODEL_ROOT": str(cache_root),
}

In [ ]:
# @title 可选 GPU Worker 模型环境
# 使用含该功能的仓库提交设置 ASMR_COLAB_REVISION，或在安装单元格中修改 REVISION。
# 普通字幕识别可跳过此单元格。下面的设置可随 worker_env 传给 GPU Worker 进程。
INSTALL_AUDIO_SEPARATOR = False  # @param {type:"boolean"}
if INSTALL_AUDIO_SEPARATOR and not cli_job_dir:
    subprocess.run([sys.executable, "-m", "pip", "install", "audio-separator[gpu]"], check=True)

# 每个可选模型使用自己的源码目录和 Python 环境。按各项目的安装说明准备后填入路径。
# Seed-VC: https://github.com/Plachtaa/seed-vc
# CosyVoice（克隆时包含子模块）: https://github.com/FunAudioLLM/CosyVoice
# GPT-SoVITS: https://github.com/RVC-Boss/GPT-SoVITS
optional_worker_settings = {
    "AUDIO_SEPARATOR_MODEL_DIR": str(cache_root / "audio_separator"),
    "ASMR_SEED_VC_ROOT": "",
    "ASMR_SEED_VC_PYTHON": "",
    "ASMR_SEED_VC_CHECKPOINT": "",
    "ASMR_SEED_VC_CONFIG": "",
    "ASMR_COSYVOICE2_ROOT": "",
    "ASMR_COSYVOICE2_PYTHON": "",
    "ASMR_COSYVOICE2_MODEL": "",
    "ASMR_GPT_SOVITS_ROOT": "",
    "ASMR_GPT_SOVITS_PYTHON": "",
    "ASMR_GPT_SOVITS_CONFIG": "",
}
if not cli_job_dir:
    worker_env.update({key: value for key, value in optional_worker_settings.items() if value})

In [ ]:
# @title 3. 读取音频并运行 Faster-Whisper
import json

if cli_job_dir:
    candidates = list(job_dir.glob("input.*"))
    if len(candidates) != 1:
        raise ValueError("任务目录中应当有且只有一个 input 音频。")
    audio_path = candidates[0]
elif AUDIO_PATH:
    audio_path = Path(AUDIO_PATH).expanduser().resolve(strict=True)
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("请上传一个音频文件。")
    name, content = next(iter(uploaded.items()))
    audio_path = job_dir / ("input" + Path(name).suffix)
    audio_path.write_bytes(content)

result_srt = job_dir / "result.srt"
job_json = job_dir / "job.json"
response_json = job_dir / "response.json"
job_json.write_text(json.dumps({
    "task": "asr",
    "request": {
        "provider": "faster_whisper",
        "model": MODEL,
        "language": LANGUAGE,
        "input_path": str(audio_path),
        "output_path": str(result_srt),
    },
    "response_path": str(response_json),
}), encoding="utf-8")
completed = subprocess.run([
    sys.executable, "-m", "packages.gpu_worker.cli", "--once", str(job_json),
], cwd=repo_dir, env=worker_env)
if completed.returncode and not response_json.is_file():
    raise RuntimeError("识别进程启动失败，请查看上方日志。")
response = json.loads(response_json.read_text(encoding="utf-8"))
if completed.returncode or not response["ok"]:
    raise RuntimeError(response.get("error", {}).get("message", "识别失败"))
segments = response["result"]["segments"]
if not segments:
    raise RuntimeError("未识别到语音，请确认语言和音频内容。")
print(f"已输出 {len(segments)} 条原文字幕：{result_srt}")

In [ ]:
# @title 4. 下载字幕
if not cli_job_dir:
    from google.colab import files
    files.download(str(result_srt))

## 可选：连接桌面端到 GPU Worker

完成 GPU、安装和可选模型设置后，可直接运行下面的服务单元格。跳过字幕识别和下载单元格也可以启动服务。使用包含所需功能的仓库版本。

Worker 默认监听 Colab 的本地端口 8765。使用桌面流水线的新版本时，请参照仓库 `docs/guides/agent-and-colab-guide.md` 第 5 节，打包当前 Worker，并通过 Colab SSH 转发本地端口。桌面端使用 `ASMR_HELPER_REMOTE_WORKER_URL` 与 `ASMR_HELPER_REMOTE_WORKER_TOKEN` 连接，音频直接传给 Worker，无需上传到 Google Drive。

In [ ]:
# @title 可选：启动 GPU Worker 服务
START_WORKER = False  # @param {type:"boolean"}
WORKER_HOST = "127.0.0.1"  # @param {type:"string"}
WORKER_PORT = 8765  # @param {type:"integer"}
if START_WORKER and not cli_job_dir:
    from getpass import getpass

    if globals().get("worker_process") and worker_process.poll() is None:
        raise RuntimeError("GPU Worker 已运行。重启前请先执行 worker_process.terminate()。")
    token = getpass("GPU Worker 密钥").strip()
    if not token or not token.isascii() or any(ch.isspace() for ch in token):
        raise ValueError("请输入一个不含空格的 ASCII 密钥。")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "fastapi>=0.115", "uvicorn>=0.34", "httpx>=0.25",
    ], check=True)
    try:
        worker_process = subprocess.Popen([
            sys.executable, "-m", "packages.gpu_worker.cli", "--serve",
            "--host", WORKER_HOST, "--port", str(WORKER_PORT),
        ], cwd=repo_dir, env={**worker_env, "ASMR_WORKER_TOKEN": token})
    finally:
        del token